# Retrieval v2 – Stage 0 / Stage 1 / Stage 2

This notebook is a thin control and inspection layer.

- **Stage 0** independently reproduces the frozen Baseline v1 retrieval behavior and must pass the strict historical per-query fixture.
- **Stage 1** evaluates L0–L4 lexical retrieval and freezes `L2_bm25_preserving` as the lexical default.
- **Stage 2** compares dense-only `E5-small`, `Qwen3-Embedding-0.6B`, and `BGE-M3` on the same four diagnostic ASR/text channels.
- Retrieval algorithms and metrics live in `src/`, not in notebook cells.
- Holdout20 remains closed.


## 1. Setup

Mount Google Drive, resolve the code root and artifact root, and verify that the Retrieval v2 files are present.

Normally no path edit is needed when the project is stored under `aic26/asr_model_comparison`. Use the override variables only when running from a different location.


In [ ]:
from pathlib import Path
import glob
import json
import os
import subprocess
import sys

import pandas as pd
from IPython.display import display

try :
    from google.colab import drive
    drive.mount("/content/drive", force_remount = False)
except Exception :
    pass


CODE_ROOT_OVERRIDE     = ""
ARTIFACT_ROOT_OVERRIDE = ""


def first_existing(candidates) :
    for candidate in candidates :
        path = Path(candidate)

        if (path.exists()) :
            return path.resolve()

    return None


def detect_artifact_root() :
    if (ARTIFACT_ROOT_OVERRIDE) :
        return Path(ARTIFACT_ROOT_OVERRIDE).expanduser().resolve()

    environment = os.environ.get("AIC_RETRIEVAL_ARTIFACT_ROOT", "").strip()

    if (environment) :
        return Path(environment).expanduser().resolve()

    candidates = [
        "/content/drive/MyDrive/aic26/asr_model_comparison",
    ]
    candidates.extend(
        glob.glob(
            "/content/drive/.shortcut-targets-by-id/*/aic26/asr_model_comparison"
        )
    )

    result = first_existing(candidates)

    if (result is None) :
        raise FileNotFoundError(
            "Could not detect the ASR artifact workspace. "
            "Set ARTIFACT_ROOT_OVERRIDE in this cell."
        )

    return result


def detect_code_root(artifact_root) :
    if (CODE_ROOT_OVERRIDE) :
        return Path(CODE_ROOT_OVERRIDE).expanduser().resolve()

    environment = os.environ.get("AIC_RETRIEVAL_CODE_ROOT", "").strip()

    if (environment) :
        return Path(environment).expanduser().resolve()

    candidates = [
        Path.cwd(),
        Path("/content/aic-2026-asr-retrieval"),
        artifact_root,
    ]

    for candidate in candidates :
        if ((candidate / "configs" / "retrieval_v2.json").exists()) :
            return candidate.resolve()

    raise FileNotFoundError(
        "Could not detect the Retrieval v2 code root. "
        "Clone/copy the repository first or set CODE_ROOT_OVERRIDE."
    )


ARTIFACT_ROOT = detect_artifact_root()
CODE_ROOT     = detect_code_root(ARTIFACT_ROOT)

os.environ["AIC_RETRIEVAL_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["AIC_RETRIEVAL_CODE_ROOT"]     = str(CODE_ROOT)

RUNNER = CODE_ROOT / "src" / "07_evaluate_retrieval_v2.py"
CONFIG = CODE_ROOT / "configs" / "retrieval_v2.json"

print("Code root    :", CODE_ROOT)
print("Artifact root:", ARTIFACT_ROOT)
print("Runner exists:", RUNNER.exists())
print("Config exists:", CONFIG.exists())

assert RUNNER.exists()
assert CONFIG.exists()


## 2. Stage selection

Stage 0 and Stage 1 must already have passed their unchanged contracts before Stage 2 runs. Adding Stage 2 config does not invalidate earlier stages when their own contracts are unchanged.


In [ ]:
STAGE = "stage02_dense"

assert STAGE in {"stage00_baseline", "stage01_lexical", "stage02_dense"}
print("Selected stage:", STAGE)


## 3. Configuration and integrity preview

This cell shows the frozen benchmark identities, channels, and retrieval policy before execution.


In [ ]:
config = json.loads(CONFIG.read_text(encoding = "utf-8"))

print("Retrieval ID :", config["retrieval_id"])
print("Frozen       :", config["retrieval_frozen"])
print("\nChannels:")
for channel_id, item in config["channels"].items() : print(f"  {channel_id:20s} model={item['model_id']:32s} view={item['view']}")

print("\nEvaluation policy:")
print(json.dumps(config["evaluation_policy"], ensure_ascii = False, indent = 2))

if (STAGE == "stage00_baseline") :
    print("\nStage 0 semantic model:"); print(json.dumps(config["stage00_baseline"]["semantic"], ensure_ascii = False, indent = 2))
elif (STAGE == "stage01_lexical") :
    print("\nStage 1 lexical methods:"); print(json.dumps(config["stage01_lexical"], ensure_ascii = False, indent = 2))
else :
    print("\nStage 2 dense backends:")
    for backend_id, item in config["stage02_dense"]["backends"].items() : print(f"  {backend_id:28s} {item['model_name']}@{item['revision'][:12]}")
    print("\nStage 2 score policy:"); print(json.dumps(config["stage02_dense"]["score"], ensure_ascii = False, indent = 2))

print("\nStrict Stage 0 fixture:")
print(json.dumps(config["regression"], ensure_ascii = False, indent = 2))


## 4. Execute Retrieval v2

The runner streams live progress, validates all frozen identities, reuses valid embedding caches, evaluates rankings, writes reports, and exits non-zero on failure.


In [ ]:
command = [sys.executable, "-u", str(RUNNER), "--stage", STAGE]
env     = {**os.environ, "PYTHONUNBUFFERED" : "1"}

print("Running:", " ".join(command), "\n")
process = subprocess.Popen(command, cwd = str(CODE_ROOT), env = env, stdout = subprocess.PIPE, stderr = subprocess.STDOUT, text = True, bufsize = 1)
for line in process.stdout : print(line, end = "", flush = True)
returncode = process.wait()
if (returncode != 0) : raise RuntimeError(f"Retrieval v2 failed with exit code {returncode}")


## 5. Stage result

Load the machine-readable stage summary and validation report.


In [ ]:
REPORT_ROOT = ARTIFACT_ROOT / config["paths"]["reports_root"] / STAGE

summary_path    = REPORT_ROOT / "stage_summary.json"
validation_path = REPORT_ROOT / "validation_summary.json"

summary    = json.loads(summary_path.read_text(encoding = "utf-8"))
validation = json.loads(validation_path.read_text(encoding = "utf-8"))

print("=" * 88)
print("STAGE RESULT")
print("=" * 88)
print("Stage :", summary["stage"])
print("Status:", "PASS" if summary["passed"] else "FAIL")
print("Reports:", REPORT_ROOT)

if ("strict_fixture" in summary) :
    print()
    print("Strict historical fixture:")
    print(json.dumps(summary["strict_fixture"], ensure_ascii = False, indent = 2))

print()
print("Validation:", "PASS" if validation["passed"] else "FAIL")


## 6. Aggregate metrics

Stage 0 reproduces Baseline v1, Stage 1 compares lexical methods, and Stage 2 compares the three dense backends independently across Whisper/Parakeet raw/processed.


In [ ]:
metrics = pd.read_csv(REPORT_ROOT / "retrieval_metrics.csv")
metric_columns = ["query_set", "method_id", "model_id", "view", "query_count", "story_recall_at_1", "story_recall_at_3", "story_recall_at_5", "story_recall_at_10", "story_mrr", "video_recall_at_1", "video_recall_at_3", "video_recall_at_5", "video_recall_at_10", "video_recall_at_20", "video_mrr"]
display(metrics[[column for column in metric_columns if column in metrics.columns]].sort_values(["query_set", "method_id", "model_id", "view"]).reset_index(drop = True))

## 7. Per-query comparison

Stage 1 compares lexical methods against L0. Stage 2 compares Qwen3 and BGE-M3 against dense control D0 E5-small and also reports the historical Baseline v1 system reference.


In [ ]:
query_results = pd.read_csv(REPORT_ROOT / "query_results.csv")

if (STAGE == "stage01_lexical") :
    method_summary, query_comparison = pd.read_csv(REPORT_ROOT / "method_summary.csv"), pd.read_csv(REPORT_ROOT / "query_comparison.csv")
    print("Method-level paired summary:"); display(method_summary)
    print("\nPer-query changes vs L0:"); display(query_comparison.head(80))
elif (STAGE == "stage02_dense") :
    method_summary, query_comparison = pd.read_csv(REPORT_ROOT / "method_summary.csv"), pd.read_csv(REPORT_ROOT / "query_comparison.csv")
    backend_summary, asr_gap_summary = pd.read_csv(REPORT_ROOT / "backend_summary.csv"), pd.read_csv(REPORT_ROOT / "asr_gap_summary.csv")
    asr_gap_queries, system_reference = pd.read_csv(REPORT_ROOT / "asr_gap_queries.csv"), pd.read_csv(REPORT_ROOT / "system_reference_comparison.csv")
    print("Dense method paired summary vs D0 E5-small:"); display(method_summary)
    print("\nBackend/runtime summary:"); display(backend_summary)
    print("\nWhisper–Parakeet aggregate gap by dense backend:"); display(asr_gap_summary)
    print("\nWhisper–Parakeet paired query gaps:"); display(asr_gap_queries.head(120))
    print("\nDense-only performance vs historical Baseline v1:"); display(system_reference)
    print("\nPer-query changes vs D0:"); display(query_comparison.head(120))
else :
    print("Stage 0 query results:")
    columns = ["query_set", "model_id", "view", "query_id", "first_relevant_rank", "video_rank", "top_story_window_id", "top_video_id", "story_score_margin", "video_score_margin"]
    display(query_results[columns].head(120))

## 8. Failure and evidence inspection

Stage 1 exposes lexical diagnostics. Stage 2 focuses on dense discrimination: score spread, top-k evidence concentration, correct-video support, and representative top windows/videos.


In [ ]:
if (STAGE == "stage01_lexical") :
    diagnostics, evidence = pd.read_csv(REPORT_ROOT / "lexical_diagnostics.csv"), pd.read_csv(REPORT_ROOT / "top_evidence.csv")
    print("Lowest lexical coverage / zero-evidence cases:"); display(diagnostics.sort_values(["zero_evidence", "query_coverage"], ascending = [False, True]).head(40))
elif (STAGE == "stage02_dense") :
    diagnostics, evidence = pd.read_csv(REPORT_ROOT / "dense_diagnostics.csv"), pd.read_csv(REPORT_ROOT / "top_evidence.csv")
    print("Worst dense video margins:"); display(diagnostics.sort_values(["video_score_margin", "video_rank"], ascending = [True, False]).head(40))
else :
    regression = json.loads((REPORT_ROOT / "regression.json").read_text(encoding = "utf-8"))
    print("Stage 0 regression summary:"); print(json.dumps(summary["regression"], ensure_ascii = False, indent = 2))
    evidence = None

if (STAGE in {"stage01_lexical", "stage02_dense"}) :
    QUERY_ID = "R2-20" if "R2-20" in set(evidence["query_id"]) else evidence["query_id"].iloc[0]
    print("\nEvidence for query:", QUERY_ID)
    display(evidence[evidence["query_id"] == QUERY_ID].sort_values(["method_id", "model_id", "view", "entity_type", "rank"]).head(160))

## 9. Research decision

Record the decision only after reviewing aggregate metrics, paired ranks, backend/runtime behavior, ASR gaps, and failure evidence. Stage 2 must select at most one dense default and one close alternative; no ASR or text-view fusion is introduced.


In [ ]:
RETRIEVAL_DECISION = {
    "stage" : STAGE,
    "lexical_default" : "L2_bm25_preserving",
    "provisional_dense_default" : None,
    "retained_dense_alternative" : None,
    "main_observation" : None,
    "failure_observation" : None,
    "asr_gap_observation" : None,
    "notes" : ["Stage 1 lexical default is frozen as L2_bm25_preserving.", "Stage 2 compares dense representations only; max-window aggregation remains frozen.", "Whisper and Parakeet remain independent diagnostic channels; no ASR fusion is planned.", "Raw and processed remain diagnostic views; no text-view fusion is assumed.", "Holdout20 remains closed."],
}
print(json.dumps(RETRIEVAL_DECISION, ensure_ascii = False, indent = 2))